# 01 — Data Collection
Pull AQI data from EPA AirNow and weather data from NOAA CDO, then save to `data/raw/`.

**Before running:** Set your API keys in a `.env` file at the project root:
```
AIRNOW_API_KEY=your-key
NOAA_API_KEY=your-token
```

In [ ]:
import sys
sys.path.append('..')

import os
from datetime import date
from dotenv import load_dotenv

load_dotenv('../.env')

from utils.airnow import fetch_date_range
from utils.noaa import find_station, fetch_weather

os.makedirs('../data/raw', exist_ok=True)

## Config — Change These

In [ ]:
ZIP_CODE  = '98040'       # Change to your city's ZIP code
START     = date(2022, 1, 1)
END       = date(2024, 12, 31)
POLLUTANT = 'PM2.5'       # Options: 'PM2.5' or 'OZONE'

## 1. Pull AirNow AQI Data
This fetches one day at a time (AirNow's historical endpoint is per-day).
3 years of data takes ~5–10 minutes. Run once, then load from CSV.

In [ ]:
aqi_df = fetch_date_range(ZIP_CODE, START, END, pollutant=POLLUTANT)
aqi_df.head(10)

In [ ]:
aqi_df.to_csv('../data/raw/aqi_raw.csv', index=False)
print(f'Saved {len(aqi_df)} rows to data/raw/aqi_raw.csv')

## 2. Pull NOAA Weather Data
Find the nearest station with good data coverage, then pull daily summaries.

In [ ]:
station_id = find_station(ZIP_CODE, START, END)
print('Station ID:', station_id)

# If None, look one up manually at: https://www.ncei.noaa.gov/cdo-web/search
# Then hardcode it: station_id = 'GHCND:USW00024233'

In [ ]:
weather_df = fetch_weather(station_id, START, END)
weather_df.head(10)

In [ ]:
weather_df.to_csv('../data/raw/weather_raw.csv', index=False)
print(f'Saved {len(weather_df)} rows to data/raw/weather_raw.csv')

## 3. Sanity Check

In [ ]:
print('AQI shape:    ', aqi_df.shape)
print('Weather shape:', weather_df.shape)
print('\nAQI date range:    ', aqi_df['date'].min(), 'to', aqi_df['date'].max())
print('Weather date range:', weather_df['date'].min(), 'to', weather_df['date'].max())
print('\nMissing AQI values:\n',     aqi_df.isnull().sum())
print('\nMissing weather values:\n', weather_df.isnull().sum())